In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

**Load the Dataset**

In [3]:
data = pd.read_csv("CSE_Skill_Gap_Dataset_1000.csv")

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (1000, 22)


,StudentName,CurriculumCoverage,IndustryDemand,JobFrequency,ExpertImportance,Programming,DataStructures,Algorithms,Database,WebDevelopment,...,Communication,ProblemSolving,Teamwork,Projects,Internship,Certifications,CurriculumAlignment,IndustryRequirement,SkillGap,GapStatus
0,Kiran Patel,74,81,2,2,58,63,46,80,41,...,79,58,48,3,0,3,76,84,8,0
1,Diya Singh,83,75,10,5,63,64,49,87,54,...,80,44,45,3,0,0,78,78,0,0
2,Ananya Verma,46,59,1,4,57,63,98,46,69,...,94,76,40,1,1,2,44,56,12,1
3,Sneha Sharma,60,91,5,1,72,94,97,83,75,...,95,72,42,3,1,1,57,86,29,1
4,Arjun Naidu,59,75,10,2,43,100,69,45,71,...,72,42,54,2,0,1,57,79,22,1


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   StudentName          1000 non-null   object
 1   CurriculumCoverage   1000 non-null   int64 
 2   IndustryDemand       1000 non-null   int64 
 3   JobFrequency         1000 non-null   int64 
 4   ExpertImportance     1000 non-null   int64 
 5   Programming          1000 non-null   int64 
 6   DataStructures       1000 non-null   int64 
 7   Algorithms           1000 non-null   int64 
 8   Database             1000 non-null   int64 
 9   WebDevelopment       1000 non-null   int64 
 10  Cloud                1000 non-null   int64 
 11  AI_ML                1000 non-null   int64 
 12  Communication        1000 non-null   int64 
 13  ProblemSolving       1000 non-null   int64 
 14  Teamwork             1000 non-null   int64 
 15  Projects             1000 non-null   int64 
 16  Interns

In [5]:
print(data.isnull().sum())

StudentName            0
CurriculumCoverage     0
IndustryDemand         0
JobFrequency           0
ExpertImportance       0
Programming            0
DataStructures         0
Algorithms             0
Database               0
WebDevelopment         0
Cloud                  0
AI_ML                  0
Communication          0
ProblemSolving         0
Teamwork               0
Projects               0
Internship             0
Certifications         0
CurriculumAlignment    0
IndustryRequirement    0
SkillGap               0
GapStatus              0
dtype: int64


**Select Relevant Features**

In [7]:
# Select Relevant Features

features = [
    "StudentName",
    "CurriculumCoverage",
    "IndustryDemand",
    "JobFrequency",
    "ExpertImportance",
    "Programming",
    "DataStructures",
    "Algorithms",
    "Database",
    "WebDevelopment",
    "Cloud",
    "AI_ML",
    "Communication",
    "ProblemSolving",
    "Teamwork",
    "Projects",
    "Internship",
    "Certifications",
    "CurriculumAlignment",
    "IndustryRequirement",
    "SkillGap"
]

# Target Variable

target = "GapStatus"

# Create Feature Matrix and Target Vector

X = data[features]
y = data[target]

# Display Features and Target

print(X.head())
print(y.head())

    StudentName  CurriculumCoverage  IndustryDemand  JobFrequency  \
0   Kiran Patel                  74              81             2   
1    Diya Singh                  83              75            10   
2  Ananya Verma                  46              59             1   
3  Sneha Sharma                  60              91             5   
4   Arjun Naidu                  59              75            10   

   ExpertImportance  Programming  DataStructures  Algorithms  Database  \
0                 2           58              63          46        80   
1                 5           63              64          49        87   
2                 4           57              63          98        46   
3                 1           72              94          97        83   
4                 2           43             100          69        45   

   WebDevelopment  ...  AI_ML  Communication  ProblemSolving  Teamwork  \
0              41  ...     58             79              58      

**Split the Dataset**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


Training records: 800
Testing records: 200


**Before Proper Preprocessing**

In [9]:
# Before Proper Preprocessing

numerical_only = [
    "CurriculumCoverage",
    "IndustryDemand",
    "JobFrequency",
    "ExpertImportance",
    "Programming",
    "DataStructures",
    "Algorithms",
    "Database",
    "WebDevelopment",
    "Cloud",
    "AI_ML",
    "Communication",
    "ProblemSolving",
    "Teamwork",
    "Projects",
    "Internship",
    "Certifications",
    "CurriculumAlignment",
    "IndustryRequirement",
    "SkillGap"
]

baseline_train = X_train[numerical_only].copy()
baseline_test = X_test[numerical_only].copy()

train_valid_rows = baseline_train.dropna().index
test_valid_rows = baseline_test.dropna().index

baseline_train = baseline_train.loc[train_valid_rows]
baseline_test = baseline_test.loc[test_valid_rows]

y_train_baseline = y_train.loc[train_valid_rows]
y_test_baseline = y_test.loc[test_valid_rows]

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(baseline_train, y_train_baseline)

baseline_predictions = baseline_model.predict(baseline_test)

baseline_accuracy = accuracy_score(
    y_test_baseline,
    baseline_predictions
)

print("Baseline accuracy:", baseline_accuracy)
print("Training records used:", len(baseline_train))
print("Testing records used:", len(baseline_test))

Baseline accuracy: 0.995
Training records used: 800
Testing records used: 200


**Define Numerical and Categorical Features**

In [10]:
# Define Numerical and Categorical Features

numerical_features = [
    "CurriculumCoverage",
    "IndustryDemand",
    "JobFrequency",
    "ExpertImportance",
    "Programming",
    "DataStructures",
    "Algorithms",
    "Database",
    "WebDevelopment",
    "Cloud",
    "AI_ML",
    "Communication",
    "ProblemSolving",
    "Teamwork",
    "Projects",
    "Internship",
    "Certifications",
    "CurriculumAlignment",
    "IndustryRequirement",
    "SkillGap"
]

categorical_features = [
    "StudentName"
]

**Numerical Preprocessing**
For numerical variables:

Replace missing values using the median.
Standardize the values.

In [11]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

**Categorical Preprocessing**

For categorical variables:

Replace missing values using the most frequent category.
Convert categories using one-hot encoding.

In [12]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

**Combine the Preprocessing Steps**

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

**Create the Complete Machine-Learning Pipeline**

In [14]:
complete_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

In [15]:
complete_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['CurriculumCoverage',
                                                   'IndustryDemand',
                                                   'JobFrequency',
                                                   'ExpertImportance',
                                                   'Programming',
                                                   'DataStructures',
                                                   'Algorithms', 'Database',
                                                   'WebDevelopment', 'Cloud',
                                                   'AI_ML', 'Communication',
                                                   'ProblemSolving', 'Teamwork',
                                                   'Projects', 'Internship',
                                                   'Certifications',
                                                   'CurriculumAlignment',
                                                   'IndustryRequirement',
                                                   'SkillGap']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['StudentName'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [16]:
processed_predictions = complete_pipeline.predict(X_test)

processed_accuracy = accuracy_score(
    y_test,
    processed_predictions
)

print("Accuracy after preprocessing:", processed_accuracy)

Accuracy after preprocessing: 0.99


In [17]:
print(
    classification_report(
        y_test,
        processed_predictions
    )
)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       107
           1       1.00      0.98      0.99        93

    accuracy                           0.99       200
   macro avg       0.99      0.99      0.99       200
weighted avg       0.99      0.99      0.99       200



**Compare Before and After Preprocessing**

In [18]:
comparison = pd.DataFrame(
    {
        "Approach": [
            "Before proper preprocessing",
            "After preprocessing pipeline"
        ],
        "Accuracy": [
            baseline_accuracy,
            processed_accuracy
        ],
        "Training records used": [
            len(baseline_train),
            len(X_train)
        ],
        "Testing records used": [
            len(baseline_test),
            len(X_test)
        ]
    }
)

print(comparison)

                       Approach  Accuracy  Training records used  \
0   Before proper preprocessing     0.995                    800   
1  After preprocessing pipeline     0.990                    800   

   Testing records used  
0                   200  
1                   200  


**Generate the Processed Dataset**

In [19]:
X_train_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

In [20]:
feature_names = complete_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

print(feature_names)

['numerical__CurriculumCoverage' 'numerical__IndustryDemand'
 'numerical__JobFrequency' 'numerical__ExpertImportance'
 'numerical__Programming' 'numerical__DataStructures'
 'numerical__Algorithms' 'numerical__Database' 'numerical__WebDevelopment'
 'numerical__Cloud' 'numerical__AI_ML' 'numerical__Communication'
 'numerical__ProblemSolving' 'numerical__Teamwork' 'numerical__Projects'
 'numerical__Internship' 'numerical__Certifications'
 'numerical__CurriculumAlignment' 'numerical__IndustryRequirement'
 'numerical__SkillGap' 'categorical__StudentName_Aarav Gupta'
 'categorical__StudentName_Aarav Kumar'
 'categorical__StudentName_Aarav Naidu'
 'categorical__StudentName_Aarav Patel'
 'categorical__StudentName_Aarav Rao'
 'categorical__StudentName_Aarav Reddy'
 'categorical__StudentName_Aarav Sharma'
 'categorical__StudentName_Aarav Singh'
 'categorical__StudentName_Aarav Verma'
 'categorical__StudentName_Aarav Yadav'
 'categorical__StudentName_Aditya Gupta'
 'categorical__StudentName_Adity

In [21]:
processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [22]:
processed_train_df["GapStatus"] = y_train.values
processed_test_df["GapStatus"] = y_test.values

In [23]:
processed_train_df.head()

,numerical__CurriculumCoverage,numerical__IndustryDemand,numerical__JobFrequency,numerical__ExpertImportance,numerical__Programming,numerical__DataStructures,numerical__Algorithms,numerical__Database,numerical__WebDevelopment,numerical__Cloud,...,categorical__StudentName_Vivaan Kumar,categorical__StudentName_Vivaan Naidu,categorical__StudentName_Vivaan Patel,categorical__StudentName_Vivaan Rao,categorical__StudentName_Vivaan Reddy,categorical__StudentName_Vivaan Sharma,categorical__StudentName_Vivaan Singh,categorical__StudentName_Vivaan Verma,categorical__StudentName_Vivaan Yadav,GapStatus
301,-0.538499,0.446489,-0.855806,-0.702822,0.318948,-1.706900,-0.236867,-0.912260,-0.501016,-0.322383,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
757,-1.519819,0.312509,-0.161440,-0.702822,0.318948,-1.538864,0.226584,0.159494,1.576607,-1.451351,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
797,-0.783829,0.446489,-0.508623,-0.702822,1.512607,-0.698681,-0.063073,-0.573811,-0.894080,0.639330,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
413,1.424140,1.518330,-1.550172,-0.005232,0.148426,-1.706900,-0.005141,-0.348179,-1.343295,0.639330,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
857,-1.151824,-0.759333,-0.855806,-0.005232,-0.874710,0.533586,-1.569286,0.949207,1.183543,1.601044,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


**Save the Processed Dataset**

In [24]:
processed_train_df.to_csv(
    "Skill_Gap_processed_train.csv",
    index=False
)

processed_test_df.to_csv(
    "Skill_Gap_processed_test.csv",
    index=False
)

In [26]:
from google.colab import files

files.download("Skill_Gap_processed_train.csv")
files.download("Skill_Gap_processed_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Save the Complete Pipeline**

In [27]:
joblib.dump(
    complete_pipeline,
    "Skill_Gap_preprocessing_model_pipeline.pkl"
)

['Skill_Gap_preprocessing_model_pipeline.pkl']

In [28]:
files.download(
    "Skill_Gap_preprocessing_model_pipeline.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Reuse the Saved Pipeline**

In [29]:
loaded_pipeline = joblib.load(
    "Skill_Gap_preprocessing_model_pipeline.pkl"
)

In [30]:
new_student = pd.DataFrame(
    {
        "StudentName": ["Meghana Reddy"],
        "CurriculumCoverage": [75],
        "IndustryDemand": [90],
        "JobFrequency": [8],
        "ExpertImportance": [5],
        "Programming": [85],
        "DataStructures": [80],
        "Algorithms": [78],
        "Database": [82],
        "WebDevelopment": [76],
        "Cloud": [74],
        "AI_ML": [88],
        "Communication": [84],
        "ProblemSolving": [90],
        "Teamwork": [86],
        "Projects": [4],
        "Internship": [1],
        "Certifications": [3],
        "CurriculumAlignment": [76],
        "IndustryRequirement": [90],
        "SkillGap": [14]
    }
)

In [31]:
prediction = loaded_pipeline.predict(new_student)

if prediction[0] == 0:
    print("Prediction: Aligned")
else:
    print("Prediction: Gap Identified")

Prediction: Gap Identified
